# 🧠 Phishing Website Detection Using ANN
### ✨ Artificial Neural Network | Deep Learning | Binary Classification

> **Goal:** Predict whether a website is **Legitimate** or **Phishing** using an Artificial Neural Network.

📌 **Model used:** ANN only  
📌 **Workflow:** Data → Cleaning → EDA → Preprocessing → ANN → Evaluation → Saving


## 1️⃣ Import Libraries

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, classification_report,
    confusion_matrix
)

import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping

print("TensorFlow version:", tf.__version__)


## 2️⃣ Load Dataset

In [ ]:
DATA_PATH = "dataset1.csv"   # Uploaded dataset

df = pd.read_csv(DATA_PATH)

print("Dataset Shape:", df.shape)
display(df.head())


## 3️⃣ Quick Dataset Understanding

In [ ]:
display(df.info())
display(df.describe(include="all").T)

print("\nMissing Values:")
display(df.isnull().sum().sort_values(ascending=False).head(15))

print("\nDuplicate Rows:", df.duplicated().sum())


## 4️⃣ Data Cleaning

In [ ]:
df = df.drop_duplicates().copy()

# Remove completely empty columns
df = df.dropna(axis=1, how="all")

print("Shape after cleaning:", df.shape)


## 5️⃣ Select Target Column

In [ ]:
# Update TARGET_COLUMN if your dataset uses another target name.
possible_targets = ["Result"]

TARGET_COLUMN = next((col for col in possible_targets if col in df.columns), None)

if TARGET_COLUMN is None:
    print("Available columns:", list(df.columns))
    raise ValueError("Set TARGET_COLUMN manually in this cell.")

print("Target column:", TARGET_COLUMN)
print(df[TARGET_COLUMN].value_counts())


## 6️⃣ Prepare Features and Target

In [ ]:
X = df.drop(columns=[TARGET_COLUMN]).copy()\n\n# Remove row index column (not a predictive feature)\nif "index" in X.columns:\n    X = X.drop(columns=["index"])
y = df[TARGET_COLUMN].copy()

# Convert categorical feature columns into numeric columns
X = pd.get_dummies(X, drop_first=True)

# Convert target labels (-1, 1) into binary labels (0, 1)
# -1 = Phishing, 1 = Legitimate in this dataset
y = y.map({-1: 0, 1: 1})

if y.isna().any():
    raise ValueError("Unexpected target labels found. Check the Result column.")

y = y.astype(int)

# Keep only numeric values and handle missing/infinite values
X = X.apply(pd.to_numeric, errors="coerce")
X = X.replace([np.inf, -np.inf], np.nan)
X = X.fillna(X.median(numeric_only=True))
X = X.fillna(0)

print("Feature shape:", X.shape)
print("Target classes:", np.unique(y))


## 7️⃣ Exploratory Data Analysis

In [ ]:
plt.figure(figsize=(7, 4))
sns.countplot(x=y)
plt.title("🎯 Target Class Distribution")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()


## 8️⃣ Train-Test Split and Scaling

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training data:", X_train_scaled.shape)
print("Testing data:", X_test_scaled.shape)


## 9️⃣ Build ANN Model

In [ ]:
tf.random.set_seed(42)

ann_model = Sequential([
    Input(shape=(X_train_scaled.shape[1],)),
    Dense(64, activation="relu"),
    Dropout(0.30),
    Dense(32, activation="relu"),
    Dropout(0.20),
    Dense(16, activation="relu"),
    Dense(1, activation="sigmoid")
])

ann_model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

ann_model.summary()


## 🔟 Train ANN Model

In [ ]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

history = ann_model.fit(
    X_train_scaled,
    y_train,
    validation_split=0.20,
    epochs=50,
    batch_size=32,
    callbacks=[early_stopping],
    verbose=1
)


## 1️⃣1️⃣ Training Performance

In [ ]:
history_df = pd.DataFrame(history.history)

plt.figure(figsize=(8, 4))
plt.plot(history_df["accuracy"], label="Training Accuracy")
plt.plot(history_df["val_accuracy"], label="Validation Accuracy")
plt.title("📈 ANN Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(history_df["loss"], label="Training Loss")
plt.plot(history_df["val_loss"], label="Validation Loss")
plt.title("📉 ANN Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.show()


## 1️⃣2️⃣ Evaluate ANN

In [ ]:
probabilities = ann_model.predict(X_test_scaled, verbose=0).ravel()
predictions = (probabilities >= 0.5).astype(int)

accuracy = accuracy_score(y_test, predictions)
precision = precision_score(y_test, predictions, zero_division=0)
recall = recall_score(y_test, predictions, zero_division=0)
f1 = f1_score(y_test, predictions, zero_division=0)
roc_auc = roc_auc_score(y_test, probabilities)

metrics_df = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score", "ROC-AUC"],
    "Score": [accuracy, precision, recall, f1, roc_auc]
})

display(metrics_df.style.format({"Score": "{:.3f}"}))
print("\nClassification Report:\n")
print(classification_report(y_test, predictions, zero_division=0))


## 1️⃣3️⃣ Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, predictions)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title("🧩 ANN Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


## 1️⃣4️⃣ Save ANN Model and Preprocessing Objects

In [ ]:
import joblib

Path("models").mkdir(exist_ok=True)

ann_model.save("models/phishing_ann_model.keras")
joblib.dump(scaler, "models/scaler.pkl")
joblib.dump(list(X.columns), "models/feature_columns.pkl")

print("✅ ANN model saved successfully!")
print("✅ Scaler saved successfully!")
print("✅ Feature columns saved successfully!")


## 🎉 Project Completed

### Final Model
**Artificial Neural Network (ANN)**

### Files Generated
- `models/phishing_ann_model.keras`
- `models/scaler.pkl`
- `models/feature_columns.pkl`

> ⚠️ Before running, make sure `dataset.csv` exists and verify the target column.
